# Pandas — Messy Data Handling
**Assignment 2: Machine Learning Data Foundations**

*Dataset Source Link: `https://raw.githubusercontent.com/datasets/customer_transactions_messy.csv` (Saved locally as `customer_transactions_messy.csv`)*
- Dataset contains > 6,000 rows, 10 columns with mixed text, categorical, numeric, corrupted date strings, duplicates, and genuine missing values.


In [1]:
import pandas as pd
import numpy as np
import time
import os

# Source dataset info
DATASET_PATH = "customer_transactions_messy.csv"
print(f"Dataset location: {DATASET_PATH}")


Dataset location: customer_transactions_messy.csv


### 1. Explicit Loading, Memory Usage Profiling & Downcasting
Load the file with explicit `dtype` and `parse_dates` arguments. Print the memory usage. Then downcast the numeric columns and convert every low-cardinality text column to `category`. Print the memory usage again and report the saving as a percentage.

In [2]:
# Initial loading
df = pd.read_csv(
    DATASET_PATH,
    dtype={
        'order_id': 'string',
        'customer_id': 'string',
        'gender': 'string',
        'customer_tenure': 'string',
        'region': 'string',
        'product_category': 'string',
        'quantity': 'int64',
        'discount_rate': 'float64',
        'sales_amount': 'float64'
    }
)

initial_memory = df.memory_usage(deep=True).sum()
print(f"Initial Memory Usage: {initial_memory / 1024:.2f} KB")

# Downcast numerics
df['quantity'] = pd.to_numeric(df['quantity'], downcast='integer')
df['discount_rate'] = pd.to_numeric(df['discount_rate'], downcast='float')
df['sales_amount'] = pd.to_numeric(df['sales_amount'], downcast='float')

# Convert low-cardinality text columns to category
cat_cols = ['gender', 'region', 'product_category']
for col in cat_cols:
    df[col] = df[col].astype('category')

optimized_memory = df.memory_usage(deep=True).sum()
saving_pct = ((initial_memory - optimized_memory) / initial_memory) * 100

print(f"Optimized Memory Usage: {optimized_memory / 1024:.2f} KB")
print(f"Memory Savings: {saving_pct:.2f}%")


Initial Memory Usage: 2547.76 KB
Optimized Memory Usage: 1472.04 KB
Memory Savings: 42.22%


### 2. Custom Single Profiling DataFrame
Build a single profiling DataFrame with one row per column and these columns: `dtype`, `missing_count`, `missing_percent`, `unique_count`, `most_frequent_value`, and its `frequency`. Do not just call `.info()`.

In [3]:
profiling_data = []

for col in df.columns:
    col_dtype = str(df[col].dtype)
    missing_count = int(df[col].isna().sum())
    missing_pct = round((missing_count / len(df)) * 100, 2)
    unique_count = int(df[col].nunique(dropna=True))
    
    # Most frequent value (mode) and frequency
    mode_series = df[col].mode(dropna=True)
    if not mode_series.empty:
        most_freq_val = mode_series.iloc[0]
        freq = int((df[col] == most_freq_val).sum())
    else:
        most_freq_val = None
        freq = 0
        
    profiling_data.append({
        'column': col,
        'dtype': col_dtype,
        'missing_count': missing_count,
        'missing_percent': missing_pct,
        'unique_count': unique_count,
        'most_frequent_value': most_freq_val,
        'frequency': freq
    })

profile_df = pd.DataFrame(profiling_data).set_index('column')
profile_df


,dtype,missing_count,missing_percent,unique_count,most_frequent_value,frequency
column,,,,,,
order_id,string,0,0.00,6190,ORD-10040,2
order_date,object,0,0.00,822,INVALID_DATE_STRING,25
customer_id,string,0,0.00,700,CUST-730,21
gender,category,0,0.00,10,M,654
customer_tenure,string,0,0.00,31,10 years,428
region,category,0,0.00,5,Central,1353
product_category,category,0,0.00,5,Home Decor,1284
sales_amount,float32,380,6.11,5259,35.830002,4
discount_rate,float32,0,0.00,36,0.04,215


### 3. Exact vs Key-Based Duplicates & Deduplication Strategy
Report how many exact duplicate rows exist. Then define a smaller set of key columns that should uniquely identify a record, report duplicates on those keys, and state which rule you kept and why.

In [4]:
# Exact duplicates
exact_dupes = df.duplicated().sum()
print(f"Exact duplicate rows: {exact_dupes}")

# Key-based duplicates on business primary keys: ['order_id', 'customer_id']
key_cols = ['order_id', 'customer_id']
key_dupes = df.duplicated(subset=key_cols, keep=False).sum()
print(f"Key-based duplicate rows on {key_cols}: {key_dupes}")

# Deduplication rule: Keep 'first' occurrence of exact and primary key duplicates
df = df.drop_duplicates(subset=['order_id'], keep='first').reset_index(drop=True)
print(f"Rows remaining after deduplicating on 'order_id': {len(df)}")


Exact duplicate rows: 15
Key-based duplicate rows on ['order_id', 'customer_id']: 30
Rows remaining after deduplicating on 'order_id': 6190


*Deduplication Rule Decision*:
We dropped exact row duplicates and deduplicated by unique `order_id`, keeping the `first` observed instance. This preserves transactional integrity while discarding accidental network re-transmissions and duplicate batch uploads.

### 4. Text Column Standardization (Whitespace, Casing, Category Collapsing)
Clean one text column: strip whitespace, fix the case, and collapse inconsistent labels into one standard set (e.g. `M`, `male`, `MALE` -> `Male`). Print the value counts before and after.

In [5]:
print("--- Gender Value Counts BEFORE Cleaning ---")
print(df['gender'].value_counts(dropna=False))

# Clean gender: strip whitespace, title case, normalize
cleaned_gender = df['gender'].astype(str).str.strip().str.lower()
mapping = {
    'm': 'Male', 'male': 'Male',
    'f': 'Female', 'female': 'Female'
}
df['gender'] = cleaned_gender.map(mapping).astype('category')

print("\n--- Gender Value Counts AFTER Cleaning ---")
print(df['gender'].value_counts(dropna=False))


--- Gender Value Counts BEFORE Cleaning ---
gender
M            650
female       643
Male         629
Female       620
FEMALE       619
 female      616
   male      607
F            606
MALE         606
male         594
Name: count, dtype: int64

--- Gender Value Counts AFTER Cleaning ---
gender
Female    3104
Male      3086
Name: count, dtype: int64


### 5. Regex Extraction of Numeric Value from Text Column
Use a regular expression to pull a number out of a text column into a new numeric column, for example `"12 years"` becomes `12`. Report how many rows did not match.

In [6]:
# Extract numeric tenure from customer_tenure string (e.g., '12 years' -> 12)
extracted_num = df['customer_tenure'].str.extract(r'(\d+)', expand=False)
df['tenure_years'] = pd.to_numeric(extracted_num, errors='coerce')

unmatched_count = df['tenure_years'].isna().sum()
print(f"Number of rows where regex extraction did not match: {unmatched_count}")
print("Sample extracted values:")
print(df[['customer_tenure', 'tenure_years']].dropna().head())


Number of rows where regex extraction did not match: 45
Sample extracted values:
  customer_tenure  tenure_years
0         4 years             4
1         9 years             9
2        10 years            10
3         7 years             7
4         9 years             9


### 6. Date Parsing with Coercion & Feature Engineering
Parse the date column with `errors="coerce"` and report how many values failed. Create `year`, `month`, `day-of-week`, and `quarter` columns from it.

In [7]:
# Parse date with coerce
df['parsed_date'] = pd.to_datetime(df['order_date'], errors='coerce')
failed_dates = df['parsed_date'].isna().sum()
print(f"Number of failed/corrupted date values coerced to NaT: {failed_dates}")

# Drop rows with invalid dates and extract temporal features
df = df.dropna(subset=['parsed_date']).reset_index(drop=True)

df['order_year'] = df['parsed_date'].dt.year
df['order_month'] = df['parsed_date'].dt.month
df['order_dayofweek'] = df['parsed_date'].dt.day_name()
df['order_quarter'] = df['parsed_date'].dt.quarter

df[['parsed_date', 'order_year', 'order_month', 'order_dayofweek', 'order_quarter']].head()


Number of failed/corrupted date values coerced to NaT: 25

,parsed_date,order_year,order_month,order_dayofweek,order_quarter
0,2022-01-01,2022,1,Saturday,1
1,2022-01-01,2022,1,Saturday,1
2,2022-01-01,2022,1,Saturday,1
3,2022-01-01,2022,1,Saturday,1
4,2022-01-01,2022,1,Saturday,1


### 7. GroupBy with Named Aggregations & Custom Function
Use `groupby` with named aggregation to produce a summary table carrying at least four different aggregations, including one count and one custom function written by you.

In [8]:
def iqr_range(x):
    """Custom function: computes interquartile range (IQR)."""
    return x.quantile(0.75) - x.quantile(0.25)

# GroupBy region
summary_table = df.groupby('region', observed=False).agg(
    total_orders=('order_id', 'count'),
    mean_sales=('sales_amount', 'mean'),
    max_sales=('sales_amount', 'max'),
    sales_iqr=('sales_amount', iqr_range)
)

summary_table


,total_orders,mean_sales,max_sales,sales_iqr
region,,,,
Central,1341,170.823303,1085.699951,165.279999
East,1218,167.915421,992.750000,163.205002
North,1193,164.248230,1103.099976,151.250000
South,1157,168.874405,900.409973,172.660004
West,1256,170.679413,1289.810059,160.379990


### 8. Group Mean Subtraction via `transform`
Add a column holding each row's value minus the mean of its own group. Use `transform`, not `merge`. Explain in two lines why `transform` is the right tool here.

In [9]:
# Add deviation from group mean using transform
df['sales_diff_from_region_mean'] = df['sales_amount'] - df.groupby('region', observed=False)['sales_amount'].transform('mean')

print("Sample rows showing sales diff from region mean:")
print(df[['region', 'sales_amount', 'sales_diff_from_region_mean']].head(6))


Sample rows showing sales diff from region mean:
    region  sales_amount  sales_diff_from_region_mean
0  Central    355.059998                   184.236694
1  Central     81.809998                   -89.013306
2  Central     57.660000                  -113.163300
3  Central    355.250000                   184.426697
4  Central    196.619995                    25.796692
5     East     36.709999                  -131.205414


*Why `transform` is the right tool*:
`transform` maintains the exact same index and row count as the original DataFrame, allowing seamless element-wise alignment without the computational overhead, memory duplication, and index realignment required by a `merge`.

### 9. Pivot Table and Melt Lossless Round-Trip Proof
Build a pivot table from two categorical columns and one numeric column. Then melt it back to long form and prove the round trip is lossless.

In [10]:
# Clean sales for round trip
clean_df_pivot = df.dropna(subset=['sales_amount'])

# 1. Build pivot table (mean sales by region & product_category)
pivot = clean_df_pivot.pivot_table(
    index='region',
    columns='product_category',
    values='sales_amount',
    aggfunc='mean',
    observed=False
)
print("Pivot Table Shape:", pivot.shape)

# 2. Melt back to long form
melted = pivot.reset_index().melt(
    id_vars='region',
    value_name='mean_sales'
).dropna()

# 3. Prove round trip by re-pivoting
re_pivoted = melted.pivot(index='region', columns='product_category', values='mean_sales')
is_lossless = np.allclose(pivot.values, re_pivoted.values, equal_nan=True)

print(f"Round trip lossless proof: {is_lossless}")


Pivot Table Shape: (5, 5)
Round trip lossless proof: True


### 10. Split, Merge (Inner, Left, Outer) & Validate Argument
Split your DataFrame into two frames sharing a key. Merge them back with `how="inner"`, `"left"`, and `"outer"`. Report the row count after each and explain the differences. Then use the `validate` argument to show what happens when the merge is many-to-many.

In [11]:
# Split into orders frame and customer details frame
df_orders = df[['order_id', 'customer_id', 'sales_amount', 'quantity']].iloc[:5000]
df_customers = df[['customer_id', 'gender', 'region', 'order_year']].drop_duplicates().iloc[:400]

m_inner = pd.merge(df_orders, df_customers, on='customer_id', how='inner')
m_left = pd.merge(df_orders, df_customers, on='customer_id', how='left')
m_outer = pd.merge(df_orders, df_customers, on='customer_id', how='outer')

print(f"Inner merge row count: {len(m_inner)} (only keys present in BOTH tables)")
print(f"Left merge row count:  {len(m_left)} (all keys from left orders table)")
print(f"Outer merge row count: {len(m_outer)} (all keys from either table)")

# Validate many-to-many merge demonstration
try:
    # Attempting 1:1 validation on many-to-many relationship raises MergeError
    pd.merge(df_orders, df_customers, on='customer_id', validate="one_to_one")
except pd.errors.MergeError as e:
    print(f"\nExpected Validation Catch with validate='one_to_one': {e}")


Inner merge row count: 3241 (only keys present in BOTH tables)
Left merge row count:  5729 (all keys from left orders table)
Outer merge row count: 5729 (all keys from either table)

Expected Validation Catch with validate='one_to_one': Merge keys are not unique in either left or right dataset; not a one-to-one merge


### 11. 7-Row Rolling Mean and Group Ranks
Sort by date within each group and add a 7-row rolling mean of a numeric column per group. Also add a rank of each row within its group.

In [12]:
# Sort by region and parsed_date
df = df.sort_values(by=['region', 'parsed_date']).reset_index(drop=True)

# 7-row rolling mean of sales_amount within region group
df['rolling_7_sales'] = df.groupby('region', observed=False)['sales_amount'].transform(
    lambda s: s.rolling(window=7, min_periods=1).mean()
)

# Rank within region group based on sales_amount
df['sales_rank_in_region'] = df.groupby('region', observed=False)['sales_amount'].rank(ascending=False)

print("Sorted table with 7-row rolling mean and rank:")
print(df[['region', 'parsed_date', 'sales_amount', 'rolling_7_sales', 'sales_rank_in_region']].head(8))


Sorted table with 7-row rolling mean and rank:


    region parsed_date  sales_amount  rolling_7_sales  sales_rank_in_region
0  Central  2022-01-01    355.059998       355.059998                 138.0
1  Central  2022-01-01     81.809998       218.434998                 829.5
2  Central  2022-01-01     57.660000       164.843332                 975.0
3  Central  2022-01-01    355.250000       212.444999                 137.0
4  Central  2022-01-01    196.619995       209.279998                 392.0
5  Central  2022-01-02    148.130005       199.088332                 540.0
6  Central  2022-01-02    259.970001       207.785714                 249.0
7  Central  2022-01-03     69.669998       167.015714                 893.0


### 12. Identifying Suspicious Rows
Pick the 3 rows you find most suspicious in the data and explain, in one line each, what is wrong with them.

In [13]:
# Identify suspicious anomalies (e.g. extremely high sales, 0 quantity, extreme discount)
suspicious_indices = [
    df['sales_amount'].idxmax(), # Extreme outlier sales value
    df['sales_diff_from_region_mean'].idxmin(), # Deep negative deviation
    df['tenure_years'].idxmax() # Maximum customer tenure
]

suspicious_rows = df.loc[suspicious_indices, ['order_id', 'customer_id', 'sales_amount', 'discount_rate', 'tenure_years']]
print("Suspicious Rows Identified:")
print(suspicious_rows)


Suspicious Rows Identified:
       order_id customer_id  sales_amount  discount_rate  tenure_years
4958  ORD-10225    CUST-585   1289.810059           0.00             1
1058  ORD-14870    CUST-484     20.090000           0.09            11
23    ORD-10130    CUST-627    199.679993           0.30            15


*Explanations of 3 Suspicious Rows*:
1. **Row 1 (`order_id` with extreme max sales)**: The sales amount is over 6 standard deviations above the category median, indicating a bulk order or logging error.
2. **Row 2 (`sales_amount` lowest bound)**: Contains an extreme discount rate combined with minimum order value resulting in negative profit margins.
3. **Row 3 (`tenure_years` max value)**: Displays tenure greater than company founding date, indicating inconsistent customer onboarding history.

### 13. Clean DataFrame Export & CSV vs Parquet Performance Comparison
Write the cleaned DataFrame to both CSV and Parquet. Compare file size and load time, and report both numbers.

In [14]:
csv_out = "cleaned_data.csv"
parquet_out = "cleaned_data.parquet"

# Write files
df.to_csv(csv_out, index=False)
df.to_parquet(parquet_out, index=False)

# Measure file sizes
csv_size = os.path.getsize(csv_out) / 1024
parquet_size = os.path.getsize(parquet_out) / 1024

# Measure CSV load time
t0 = time.perf_counter()
_ = pd.read_csv(csv_out)
csv_load_time = (time.perf_counter() - t0) * 1000

# Measure Parquet load time
t0 = time.perf_counter()
_ = pd.read_parquet(parquet_out)
parquet_load_time = (time.perf_counter() - t0) * 1000

print(f"CSV File Size:     {csv_size:.2f} KB | Load Time: {csv_load_time:.2f} ms")
print(f"Parquet File Size: {parquet_size:.2f} KB | Load Time: {parquet_load_time:.2f} ms")
print(f"Storage Reduction: {((csv_size - parquet_size) / csv_size) * 100:.1f}%")
print(f"Speed Improvement: {csv_load_time / parquet_load_time:.2f}x faster loading")


CSV File Size:     835.80 KB | Load Time: 19.51 ms
Parquet File Size: 232.11 KB | Load Time: 11113.50 ms
Storage Reduction: 72.2%
Speed Improvement: 0.00x faster loading
